## Project Restructure: Descriptive Analysis

**Notes**
3 Main areas of analysis:
 * Temporal
 * Institutional
 * Geographic

Did not include all territories and jurisdictions that analysis will be done separately. Focusing solely on the U.S. states.

## Temporal Analysis

**Question**
How has undergraduate enrollment changed across U.S. states from 2012-2022, and are these changes statistically significant?

In [111]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t
import plotly.express as px
import plotly.graph_objects as go
import os
import pymannkendall as mk
from plotly.subplots import make_subplots

In [112]:
enrollment_df = pd.read_csv(os.path.dirname(os.getcwd()) + '/data/cleaned_enrollmentdata.csv')
enrollment_df.rename(columns={'State or jurisdiction': 'State'}, inplace=True)
enrollment_df.head()

,State,Year,Total_Pub_Under,4y_Pub_Under,2y_Pub_Under,Total_Pub_Postbacc,Total_Priv_Under,Np_4y_Priv_Under,Fp_4y_Priv_Under,Np_2y_Priv_Under,Fp_2y_Priv_Under,Total_Priv_Postbacc,Np_4y_Priv_Postbacc,Fp_4y_Priv_Postbacc
0,Alabama,2012-13,216535,130260,86275,34510,49382,21674,24045,518,3145,9884,3917,5967
1,Alabama,2013-14,213669,129045,84624,34615,47519,20808,23295,472,2944,9909,3866,6043
2,Alabama,2014-15,212458,129916,82542,34531,47172,21160,22714,518,2780,10867,4354,6513
3,Alabama,2015-16,212968,131503,81465,34483,44682,20938,21000,436,2308,10826,4593,6233
4,Alabama,2016-17,216115,135080,81035,34923,41893,19891,19627,386,1989,11121,4679,6442


In [113]:
states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut",
    "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
    "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan",
    "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
    "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio",
    "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia",
    "Wisconsin", "Wyoming", "District of Columbia"
]

state_enrollment = enrollment_df[enrollment_df["State"].isin(states)]
state_enrollment['State'].unique()

array(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California',
       'Colorado', 'Connecticut', 'Delaware', 'District of Columbia',
       'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana',
       'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland',
       'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
       'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
       'New Jersey', 'New Mexico', 'New York', 'North Carolina',
       'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania',
       'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee',
       'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
       'West Virginia', 'Wisconsin', 'Wyoming'], dtype=object)

--------

**Sub-Question** What is the national enrollment trend when aggregating across all states?

In [114]:
state_enrollment['Total_Enrollment'] = state_enrollment['Total_Pub_Under'] + state_enrollment['Total_Priv_Under']
state_enrollment.head()

/var/folders/6g/_wpr8bs53rs84lqb6nn_rk900000gn/T/ipykernel_59561/1575148348.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,State,Year,Total_Pub_Under,4y_Pub_Under,2y_Pub_Under,Total_Pub_Postbacc,Total_Priv_Under,Np_4y_Priv_Under,Fp_4y_Priv_Under,Np_2y_Priv_Under,Fp_2y_Priv_Under,Total_Priv_Postbacc,Np_4y_Priv_Postbacc,Fp_4y_Priv_Postbacc,Total_Enrollment
0,Alabama,2012-13,216535,130260,86275,34510,49382,21674,24045,518,3145,9884,3917,5967,265917
1,Alabama,2013-14,213669,129045,84624,34615,47519,20808,23295,472,2944,9909,3866,6043,261188
2,Alabama,2014-15,212458,129916,82542,34531,47172,21160,22714,518,2780,10867,4354,6513,259630
3,Alabama,2015-16,212968,131503,81465,34483,44682,20938,21000,436,2308,10826,4593,6233,257650
4,Alabama,2016-17,216115,135080,81035,34923,41893,19891,19627,386,1989,11121,4679,6442,258008


In [115]:
nationwide_trends = state_enrollment.groupby('Year').agg({'Total_Enrollment': 'sum', 'Total_Pub_Under': 'sum', 'Total_Priv_Under': 'sum'}).reset_index()
nationwide_trends

,Year,Total_Enrollment,Total_Pub_Under,Total_Priv_Under
0,2012-13,17717229,13458541,4258688
1,2013-14,17459865,13332032,4127833
2,2014-15,17278075,13230125,4047950
3,2015-16,17021992,13130934,3891058
4,2016-17,16854168,13126042,3728126
5,2017-18,16745071,13085693,3659378
6,2018-19,16594731,13033822,3560909
7,2019-20,16549691,12986168,3563523
8,2020-21,15836385,12305625,3530760
9,2021-22,15433062,11929275,3503787


In [116]:
baseline_year = nationwide_trends[nationwide_trends['Year'] == '2012-13']
baseline_enrollment = nationwide_trends['Total_Enrollment'][0]
nationwide_trends['enrollment_index'] = (nationwide_trends['Total_Enrollment'] / baseline_enrollment) * 100
nationwide_trends['yoy_change'] = nationwide_trends['Total_Enrollment'].pct_change() * 100

In [117]:
total_change_pct = ((nationwide_trends['Total_Enrollment'][9] - nationwide_trends['Total_Enrollment'][0])/baseline_enrollment) * 100
print(total_change_pct)

-12.892349023653754


In [118]:
years = np.arange(len(nationwide_trends))
slope,intercept, r_value, p_value, std_err = stats.linregress(years, nationwide_trends['enrollment_index'])

In [119]:
n = len(years)
dof = n - 2
t_crit = t.ppf(0.975, dof)  # 95% CI
ci_slope = t_crit * std_err

In [120]:
print('Baseline: ', baseline_enrollment)
print('2021-22: ', nationwide_trends['Total_Enrollment'][9])
print('Total Change %: ', total_change_pct)
print('Avg Change %: ', nationwide_trends['yoy_change'].mean())
print('Slope: ', slope)
print('Intercept: ', intercept)
print('95% CI: ', ci_slope)
print('R Squared: ', r_value**2)
print('P-Value: ', p_value)

Baseline:  17717229
2021-22:  15433062
Total Change %:  -12.892349023653754
Avg Change %:  -1.5150317578493226
Slope:  -1.2641234955277412
Intercept:  100.22380669942542
95% CI:  0.2766016400003222
R Squared:  0.9328116061637491
P-Value:  5.728672704293977e-06


P-Value is much less than the alpha value of 0.05, this means that the national enrollment trend is statistically significant.

### Visualization

In [121]:
## Enrollment Over time
px.line(nationwide_trends, x='Year', y='Total_Enrollment')

In [122]:
## YoY Change Over time
px.line(nationwide_trends, x='Year', y='yoy_change')

In [123]:
## Statistical Predictions
fig = go.Figure()

fig.add_trace(go.Scatter(
    x = nationwide_trends['Year'],
    y = nationwide_trends['enrollment_index'],
    mode = 'lines',
    name = 'Enrollment Index'
))

predictions = slope * years + intercept
fig.add_trace(go.Scatter(
    x=nationwide_trends['Year'],
    y=predictions,
    mode='lines',
    name=f'Linear Trend (R_squared={r_value**2:.3f})',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title='Enrollment Index Over Time',
    xaxis_title='Year',
    yaxis_title='Enrollment Index',
    showlegend=True
)


fig.show()

Baseline is at 100. We see a clear downward trend of enrollment. The linear regression easily captures 93 % of the variance.

**Question** What is the national enrollment trend when aggregating across all states?

**Answer** There is a negative enrollment trend when aggregfating across all the states.

---------

**Question** Which states have statistically significant declining trends?

In [124]:
state_trends = state_enrollment.groupby(['State', 'Year']).agg({'Total_Enrollment': 'sum', 'Total_Pub_Under': 'sum', 'Total_Priv_Under': 'sum'}).reset_index()
state_trends

,State,Year,Total_Enrollment,Total_Pub_Under,Total_Priv_Under
0,Alabama,2012-13,265917,216535,49382
1,Alabama,2013-14,261188,213669,47519
2,Alabama,2014-15,259630,212458,47172
3,Alabama,2015-16,257650,212968,44682
4,Alabama,2016-17,258008,216115,41893
...,...,...,...,...,...
505,Wyoming,2017-18,30409,29945,464
506,Wyoming,2018-19,30058,30020,38
507,Wyoming,2019-20,29931,29678,253
508,Wyoming,2020-21,28456,27871,585


In [125]:
states = state_enrollment['State'].unique()

total_change_pct_dict = {state: [] for state in states}
baseline_enrollment_dict = {state: [] for state in states}

for state in states:
    mask = state_trends['State'] == state
    df = state_trends[state_trends['State'] == state].copy()
    year_baseline = '2012-13'
    state_baseline = df[df['Year'] == year_baseline]['Total_Enrollment'].values[0]
    baseline_enrollment_dict[state].append(state_baseline)
    total_change_pct_dict[state].append(((state_trends[state_trends['State'] == state]['Total_Enrollment'].values[9] - state_trends[state_trends['State'] == state]['Total_Enrollment'].values[0])/state_baseline) * 100)
    df['enrollment_index'] = (df['Total_Enrollment'] / state_baseline) * 100
    df['yoy_change'] = df['Total_Enrollment'].pct_change() * 100
    state_trends.loc[mask, 'enrollment_index'] = df['enrollment_index']
    state_trends.loc[mask, 'yoy_change'] = df['yoy_change']

state_trends.head()

,State,Year,Total_Enrollment,Total_Pub_Under,Total_Priv_Under,enrollment_index,yoy_change
0,Alabama,2012-13,265917,216535,49382,100.000000,NaN
1,Alabama,2013-14,261188,213669,47519,98.221626,-1.778374
2,Alabama,2014-15,259630,212458,47172,97.635728,-0.596505
3,Alabama,2015-16,257650,212968,44682,96.891135,-0.762624
4,Alabama,2016-17,258008,216115,41893,97.025764,0.138948


In [126]:
avg_change_pct_dict = {state: [] for state in states}
slope_dict = {state: [] for state in states}
intercept_dict = {state: [] for state in states}
ci_slope_dict = {state: [] for state in states}
r_squared_dict = {state: [] for state in states}
p_value_dict = {state: [] for state in states}


state_years = np.arange(len(state_trends[state_trends['State'] == 'Alabama']))

for state in states:
    slope,intercept, r_value, p_value, std_err = stats.linregress(state_years, state_trends[state_trends['State'] == state]['enrollment_index'])
    ci_slope = t.interval(0.95, len(state_years)-1, loc=slope, scale=std_err)
    slope_dict[state].append(slope)
    intercept_dict[state].append(intercept)
    ci_slope_dict[state].append(ci_slope)
    r_squared_dict[state].append(r_value**2)
    p_value_dict[state].append(p_value)
    avg_change_pct_dict[state].append(state_trends[state_trends['State'] == state]['yoy_change'].mean())

In [127]:
significant = []
insignificant = []
def print_state_info(state):
    for state in states:
        print("STATE: ", state)
        print('Baseline: ', baseline_enrollment_dict.get(state)[0])
        print('2021-22: ', state_trends[state_trends['State'] == state]['Total_Enrollment'].values[9])
        print('Total Change %: ', total_change_pct_dict[state])
        print('Avg Change %: ', avg_change_pct_dict.get(state)[0])
        print('Slope: ', slope_dict.get(state)[0])
        print('Intercept: ', intercept_dict.get(state)[0])
        print('95% CI: ', ci_slope_dict.get(state)[0])
        print('R Squared: ', r_squared_dict.get(state)[0])
        print('P-Value: ', p_value_dict.get(state)[0])
        print('_______________________________________')
        print('')
for state in states:    
    if p_value_dict.get(state)[0] < 0.05:
        significant.append(state)
    else:
        insignificant.append(state)

In [128]:
print('Significant: ', significant)
print('Not Significant: ', insignificant)

Significant:  ['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'Connecticut', 'District of Columbia', 'Florida', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']
Not Significant:  ['California', 'Colorado', 'Delaware', 'Georgia', 'Nevada', 'Texas']


The states that show not statistically significant declining trend are ['California', 'Colorado', 'Delaware', 'Georgia', 'Nevada', 'Texas']. This as to do with their p value being greater than 0.05.

In [129]:
for state in significant:
    if slope_dict.get(state)[0] > 0:
        print(state + ': ', slope_dict.get(state)[0])

District of Columbia:  1.5156406938205302
Idaho:  1.6604377727889996
New Hampshire:  16.603854933943296
Utah:  5.219142158960169


Of the states with significant trends, only 4 have a positive trend, being D.C., Idaho, New Hampshire and Utah.

#### Visualization

In [130]:
## Positive Enrollment Trend
fig = go.Figure()

for state in ['District of Columbia', 'Idaho', 'New Hampshire', 'Utah']:
    fig.add_trace(go.Scatter(x=state_trends[state_trends['State'] == state]['Year'], y=state_trends[state_trends['State'] == state]['enrollment_index'], mode='lines', name=state))

fig.update_layout(title='Enrollment Index Over Time', xaxis_title='Year', yaxis_title='Total Enrollment')
fig.show()

In [131]:
## Negative Enrollment Trend
fig = go.Figure()

for state in states:
    if state not in ['District of Columbia', 'Idaho', 'New Hampshire', 'Utah']:
        fig.add_trace(go.Scatter(x=state_trends[state_trends['State'] == state]['Year'], y=state_trends[state_trends['State'] == state]['enrollment_index'], mode='lines', name=state))

fig.update_layout(title='Enrollment Index Over Time', xaxis_title='Year', yaxis_title='Total Enrollment')
fig.show()

In [132]:
## Not Significant Trends
fig = go.Figure()

for state in insignificant:
    fig.add_trace(go.Scatter(x=state_trends[state_trends['State'] == state]['Year'], y=state_trends[state_trends['State'] == state]['enrollment_index'], mode='lines', name=state))

fig.update_layout(title='Enrollment Index Over Time', xaxis_title='Year', yaxis_title='Total Enrollment')
fig.show()

**Question** which states have statistically significant declining trends?

**Answer** 'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'Connecticut', 'Florida', 'Hawaii', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming'

These states have statistically significant negative trends making them definitively decreasing for a cause other than randomness.

---------

**sub-question** Which states show statistically significant growth trends?

In [133]:
state_enrollment.head()

,State,Year,Total_Pub_Under,4y_Pub_Under,2y_Pub_Under,Total_Pub_Postbacc,Total_Priv_Under,Np_4y_Priv_Under,Fp_4y_Priv_Under,Np_2y_Priv_Under,Fp_2y_Priv_Under,Total_Priv_Postbacc,Np_4y_Priv_Postbacc,Fp_4y_Priv_Postbacc,Total_Enrollment
0,Alabama,2012-13,216535,130260,86275,34510,49382,21674,24045,518,3145,9884,3917,5967,265917
1,Alabama,2013-14,213669,129045,84624,34615,47519,20808,23295,472,2944,9909,3866,6043,261188
2,Alabama,2014-15,212458,129916,82542,34531,47172,21160,22714,518,2780,10867,4354,6513,259630
3,Alabama,2015-16,212968,131503,81465,34483,44682,20938,21000,436,2308,10826,4593,6233,257650
4,Alabama,2016-17,216115,135080,81035,34923,41893,19891,19627,386,1989,11121,4679,6442,258008


Using the Mann-Kendall test for this

In [134]:
mk_results = []

for state in sorted(state_enrollment['State'].unique()):
    state_data = state_enrollment[state_enrollment['State'] == state].sort_values('Year')
    
    # Skip states with insufficient data
    if len(state_data) < 3:
        print(f"⚠ Skipping {state}: Insufficient data points ({len(state_data)})")
        continue
    
    values = state_data['Total_Enrollment'].values
    years_numeric = np.arange(len(values))
    
    # Mann-Kendall test
    mk_result = mk.original_test(values)
    
    # Linear regression for additional context
    slope, intercept, r_value, p_value_lr, std_err = stats.linregress(years_numeric, values)
    
    # Calculate percentage change metrics
    first_value = values[0]
    last_value = values[-1]
    total_change_pct = ((last_value - first_value) / first_value) * 100
    annual_change_pct = (slope / first_value) * 100
    
    # Determine if significant and direction
    is_significant = mk_result.p < 0.05
    trend_direction = mk_result.trend  # 'increasing', 'decreasing', or 'no trend'
    
    mk_results.append({
        'State': state,
        'MK_Trend': trend_direction,
        'MK_P_Value': mk_result.p,
        'MK_Z_Score': mk_result.z,
        'MK_Tau': mk_result.Tau,
        'Significant_at_05': is_significant,
        'Total_Change_Pct': total_change_pct,
        'Annual_Change_Pct': annual_change_pct,
        'LR_Slope': slope,
        'LR_P_Value': p_value_lr,
        'R_Squared': r_value**2,
        'First_Year_Enrollment': first_value,
        'Last_Year_Enrollment': last_value,
        'N_Years': len(values)
    })

# Convert to DataFrame
mk_df = pd.DataFrame(mk_results)

print(f"\n✓ Analyzed {len(mk_df)} states")
print(f"✓ Mann-Kendall test completed for all states")


✓ Analyzed 51 states
✓ Mann-Kendall test completed for all states


In [135]:
growth_states = mk_df[
    (mk_df['Significant_at_05'] == True) & 
    (mk_df['MK_Trend'] == 'increasing')
].sort_values('Total_Change_Pct', ascending=False)

if len(growth_states) > 0:
    print(f"\n✓ {len(growth_states)} states show statistically significant GROWTH\n")
    
    # Create nice display table
    display_cols = [
        'State', 
        'Total_Change_Pct', 
        'Annual_Change_Pct',
        'MK_P_Value',
        'R_Squared'
    ]
    
    print(growth_states[display_cols].to_string(index=False))
    
    # Summary statistics for growth states
    print(f"\n" + "-"*80)
    print("GROWTH STATES SUMMARY STATISTICS:")
    print("-"*80)
    print(f"Mean total change: {growth_states['Total_Change_Pct'].mean():.2f}%")
    print(f"Median total change: {growth_states['Total_Change_Pct'].median():.2f}%")
    print(f"Range: {growth_states['Total_Change_Pct'].min():.2f}% to {growth_states['Total_Change_Pct'].max():.2f}%")
    print(f"Mean annual growth rate: {growth_states['Annual_Change_Pct'].mean():.2f}%")
    
    # Strongest growth state
    strongest = growth_states.iloc[0]
    print(f"\nSTRONGEST GROWTH: {strongest['State']}")
    print(f"   Total change: {strongest['Total_Change_Pct']:.2f}%")
    print(f"   From {strongest['First_Year_Enrollment']:,.0f} to {strongest['Last_Year_Enrollment']:,.0f} students")
    print(f"   P-value: {strongest['MK_P_Value']:.4f}")
    
else:
    print("\n⚠ NO states show statistically significant growth at α = 0.05")
    print("\nLet's check for marginal growth (p < 0.10):")
    
    marginal_growth = mk_df[
        (mk_df['MK_P_Value'] < 0.10) & 
        (mk_df['MK_Trend'] == 'increasing')
    ].sort_values('Total_Change_Pct', ascending=False)
    
    if len(marginal_growth) > 0:
        print(f"\n{len(marginal_growth)} states show marginally significant growth (p < 0.10):\n")
        display_cols = ['State', 'Total_Change_Pct', 'MK_P_Value']
        print(marginal_growth[display_cols].to_string(index=False))
    else:
        print("No states show even marginal growth trends.")


✓ 4 states show statistically significant GROWTH

               State  Total_Change_Pct  Annual_Change_Pct  MK_P_Value  R_Squared
       New Hampshire        144.707204          16.603855    0.000083   0.995941
                Utah         37.106160           5.219142    0.000347   0.953748
               Idaho         17.320147           1.660438    0.020045   0.500452
District of Columbia         12.023313           1.515641    0.000677   0.881132

--------------------------------------------------------------------------------
GROWTH STATES SUMMARY STATISTICS:
--------------------------------------------------------------------------------
Mean total change: 52.79%
Median total change: 27.21%
Range: 12.02% to 144.71%
Mean annual growth rate: 6.25%

STRONGEST GROWTH: New Hampshire
   Total change: 144.71%
   From 66,770 to 163,391 students
   P-value: 0.0001


In [136]:
trend_summary = mk_df.groupby(['MK_Trend', 'Significant_at_05']).size().reset_index(name='Count')

print("\nTrend Classification:")
for _, row in trend_summary.iterrows():
    sig_text = "Significant" if row['Significant_at_05'] else "Not Significant"
    print(f"  {row['MK_Trend'].capitalize():12} ({sig_text:15}): {row['Count']:2} states")

# Statistical test: Are growth states significantly fewer than decline states?
sig_growth = len(mk_df[(mk_df['Significant_at_05']) & (mk_df['MK_Trend'] == 'increasing')])
sig_decline = len(mk_df[(mk_df['Significant_at_05']) & (mk_df['MK_Trend'] == 'decreasing')])

print(f"\nSignificant Growth vs Decline:")
print(f"  Growth:  {sig_growth} states")
print(f"  Decline: {sig_decline} states")
print(f"  Ratio:   {sig_decline}:{sig_growth}")

if sig_growth + sig_decline > 0:
    from scipy.stats import binomtest
    # Test if growth proportion is significantly less than 50%
    binom_result = binomtest(sig_growth, sig_growth + sig_decline, 0.5, alternative='less')
    print(f"  Binomial test p-value: {binom_result.pvalue:.4f}")
    print(f"  Conclusion: Growth states are {'significantly' if binom_result.pvalue < 0.05 else 'not significantly'} less common than decline states")



Trend Classification:
  Decreasing   (Significant    ): 40 states
  Increasing   (Significant    ):  4 states
  No trend     (Not Significant):  7 states

Significant Growth vs Decline:
  Growth:  4 states
  Decline: 40 states
  Ratio:   40:4
  Binomial test p-value: 0.0000
  Conclusion: Growth states are significantly less common than decline states


In [137]:
fig1 = go.Figure()

# Significant growth (green)
sig_growth_data = mk_df[(mk_df['Significant_at_05']) & (mk_df['MK_Trend'] == 'increasing')]
fig1.add_trace(go.Scatter(
    x=sig_growth_data['Annual_Change_Pct'],
    y=sig_growth_data['Total_Change_Pct'],
    mode='markers+text',
    name='Significant Growth',
    marker=dict(size=12, color='green', symbol='triangle-up'),
    text=sig_growth_data['State'],
    textposition='top center',
    hovertemplate='<b>%{text}</b><br>Annual: %{x:.2f}%<br>Total: %{y:.2f}%<extra></extra>'
))

# Significant decline (red)
sig_decline_data = mk_df[(mk_df['Significant_at_05']) & (mk_df['MK_Trend'] == 'decreasing')]
fig1.add_trace(go.Scatter(
    x=sig_decline_data['Annual_Change_Pct'],
    y=sig_decline_data['Total_Change_Pct'],
    mode='markers',
    name='Significant Decline',
    marker=dict(size=10, color='red', symbol='circle'),
    text=sig_decline_data['State'],
    hovertemplate='<b>%{text}</b><br>Annual: %{x:.2f}%<br>Total: %{y:.2f}%<extra></extra>'
))

# No significant trend (gray)
no_trend_data = mk_df[~mk_df['Significant_at_05']]
fig1.add_trace(go.Scatter(
    x=no_trend_data['Annual_Change_Pct'],
    y=no_trend_data['Total_Change_Pct'],
    mode='markers',
    name='Not Significant',
    marker=dict(size=8, color='lightgray', symbol='circle', opacity=0.5),
    text=no_trend_data['State'],
    hovertemplate='<b>%{text}</b><br>Annual: %{x:.2f}%<br>Total: %{y:.2f}%<extra></extra>'
))

fig1.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="No Change")
fig1.add_vline(x=0, line_dash="dash", line_color="black")

fig1.update_layout(
    title=f"State Enrollment Trends: Mann-Kendall Significance Test<br>" +
          f"<sub>{sig_growth} states growing significantly, {sig_decline} declining significantly (α=0.05)</sub>",
    xaxis_title="Average Annual Change (%)",
    yaxis_title="Total Change (%)",
    height=600,
    width=1000,
    showlegend=True,
    hovermode='closest'
)

fig1.show()

# Visualization 2: Distribution of p-values by trend direction
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Growth Trends", "Decline Trends"),
    specs=[[{"type": "histogram"}, {"type": "histogram"}]]
)

growth_trends = mk_df[mk_df['MK_Trend'] == 'increasing']
decline_trends = mk_df[mk_df['MK_Trend'] == 'decreasing']

fig2.add_trace(
    go.Histogram(
        x=growth_trends['MK_P_Value'],
        nbinsx=20,
        marker_color='green',
        name='Growth'
    ),
    row=1, col=1
)

fig2.add_trace(
    go.Histogram(
        x=decline_trends['MK_P_Value'],
        nbinsx=20,
        marker_color='red',
        name='Decline'
    ),
    row=1, col=2
)

# Add significance threshold line
fig2.add_vline(x=0.05, line_dash="dash", line_color="black", row=1, col=1)
fig2.add_vline(x=0.05, line_dash="dash", line_color="black", row=1, col=2,
               annotation_text="α=0.05")

fig2.update_xaxes(title_text="P-Value", row=1, col=1)
fig2.update_xaxes(title_text="P-Value", row=1, col=2)
fig2.update_yaxes(title_text="Number of States", row=1, col=1)

fig2.update_layout(
    title_text="Distribution of P-Values by Trend Direction",
    showlegend=False,
    height=400,
    width=1000
)

fig2.show()

# Visualization 3: If we have growth states, show their trends over time
if len(growth_states) > 0:
    fig3 = go.Figure()
    
    for _, state_row in growth_states.iterrows():
        state = state_row['State']
        state_data = state_enrollment[state_enrollment['State'] == state].sort_values('Year')
        
        fig3.add_trace(go.Scatter(
            x=state_data['Year'],
            y=state_data['Total_Enrollment'],
            mode='lines+markers',
            name=f"{state} ({state_row['Total_Change_Pct']:.1f}%)",
            hovertemplate='%{y:,.0f} students<extra></extra>'
        ))
    
    fig3.update_layout(
        title=f"Enrollment Trends: States with Significant Growth<br>" +
              f"<sub>Mann-Kendall test p < 0.05</sub>",
        xaxis_title="Academic Year",
        yaxis_title="Total Enrollment",
        height=500,
        width=1000,
        hovermode='x unified',
        showlegend=True
    )
    
    fig3.show()

**Question:** Which states show statistically growth trends in enrollment?

**Answer:** New Hampshire, Utah, Idaho, District of Columbia

----

**sub-question** What is the effect size of enrollment changes (Cohen's d)?

In [138]:
total_en_year = state_enrollment.groupby(['Year']).agg({
    'Total_Enrollment': 'sum'
}).reset_index()

In [139]:
total_en_year

,Year,Total_Enrollment
0,2012-13,17717229
1,2013-14,17459865
2,2014-15,17278075
3,2015-16,17021992
4,2016-17,16854168
5,2017-18,16745071
6,2018-19,16594731
7,2019-20,16549691
8,2020-21,15836385
9,2021-22,15433062


In [140]:
std = total_en_year['Total_Enrollment'].std()
mean = total_en_year['Total_Enrollment'].mean()
ref = total_en_year['Total_Enrollment'][0]

cohen_d = np.abs((mean - ref) / std)

In [141]:
print(cohen_d)

1.3790244595349892


This cohens d is approximately 1.38 which is a large effect size. Without the absolute value we know that the cohens d is negative which tells us that the effect size is quite large and negative meaning that it is structurally significant that the enrollment has decreased over the years.

**Question:** What is the effect size of enrollment changes (Cohen's d)?

**Answer:** 1.38

------

**sub-question** Are there inflection points where enrollment trends shifted direction?

We will utilize calculus to find inflection points.

In [142]:
enrollment_trends = state_enrollment.groupby(['Year']).agg({
    'Total_Enrollment': 'sum'
}).reset_index()

In [143]:
enrollment_trends['first_diff'] = enrollment_trends['Total_Enrollment'].diff()
enrollment_trends['second_diff'] = enrollment_trends['first_diff'].diff()
enrollment_trends['third_diff'] = enrollment_trends['second_diff'].diff()

In [144]:
enrollment_trends['infl'] = np.sign(enrollment_trends['third_diff'].fillna(0)) != 0

infl_pts = enrollment_trends[enrollment_trends['infl']]

In [145]:
infl_pts

,Year,Total_Enrollment,first_diff,second_diff,third_diff,infl
3,2015-16,17021992,-256083.0,-74293.0,-149867.0,True
4,2016-17,16854168,-167824.0,88259.0,162552.0,True
5,2017-18,16745071,-109097.0,58727.0,-29532.0,True
6,2018-19,16594731,-150340.0,-41243.0,-99970.0,True
7,2019-20,16549691,-45040.0,105300.0,146543.0,True
8,2020-21,15836385,-713306.0,-668266.0,-773566.0,True
9,2021-22,15433062,-403323.0,309983.0,978249.0,True


In [146]:
fig = px.line(enrollment_trends, x='Year', y='Total_Enrollment')
fig.add_scatter(x=infl_pts['Year'], y=infl_pts['Total_Enrollment'], mode='markers', name='Inflection Points')
fig.update_layout(
    title="Enrollment Over Time",
    xaxis_title="Academic Year",
    yaxis_title="Total Enrollment",
    height=500,
    width=1000
)

fig.show()

**Question:** Are there inflection points where enrollment trends shifted direction?

**Answer:** Yes. The inflection points are in several areas most prominently howver is the downward shift in 2020 likely due to COVID-19.